# Polymarket Political Arbitrage Research
**Goal:** Find logically-constrained market pairs where `P(A AND B) > P(A)` or `P(A AND B) > P(B)` — which is a mathematical impossibility and therefore a risk-free arbitrage.

**Live arb found today (2026-04-16):**
> Blue Wave requires `House(D) AND Senate(D)`.  
> So `P(Blue Wave) ≤ min(P(House Dem), P(Senate Dem))`.  
> But market prices: Blue Wave = **0.875**, House Dem = **0.845** → Wave > House Dem by **+3pp** 🚨

**Signal path:** this notebook → identify daily mispricings → feed into `signals/` for systematic backtesting.

In [1]:
import sys, json, requests, pandas as pd
from datetime import datetime, timezone

sys.path.insert(0, r"c:\Personal\Business & Investments\Python codes\sfera")

GAMMA    = "https://gamma-api.polymarket.com"
CLOB     = "https://clob.polymarket.com"
DATA_API = "https://data-api.polymarket.com"
AS_OF    = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
print(f"Loaded. As of: {AS_OF}")

Loaded. As of: 2026-04-16 09:39 UTC


In [10]:
def fetch_event(slug: str) -> dict:
    r = requests.get(f"{GAMMA}/events", params={"slug": slug}, timeout=15)
    r.raise_for_status()
    data = r.json()
    return data[0] if data else {}

def fetch_markets_by_tag(tag: str, limit: int = 200) -> list[dict]:
    """Fetch active markets filtered by tag (e.g. 'Politics')."""
    r = requests.get(
        f"{GAMMA}/markets",
        params={"tag": tag, "active": "true", "closed": "false", "limit": limit},
        timeout=20,
    )
    r.raise_for_status()
    return r.json()

def market_summary(mkt: dict) -> dict:
    outcomes    = json.loads(mkt.get("outcomes", "[]"))
    out_prices  = json.loads(mkt.get("outcomePrices", "[]"))
    token_ids   = json.loads(mkt.get("clobTokenIds", "[]"))
    price_map   = dict(zip(outcomes, [float(p) for p in out_prices]))
    return {
        "conditionId":   mkt.get("conditionId"),
        "question":      mkt.get("question", ""),
        "end_date":      mkt.get("endDate", ""),
        "volume":        float(mkt.get("volume") or 0),
        "liquidity":     float(mkt.get("liquidity") or 0),
        "best_bid":      float(mkt.get("bestBid") or 0),
        "best_ask":      float(mkt.get("bestAsk") or 0),
        "last_price":    float(mkt.get("lastTradePrice") or list(price_map.values())[0] if price_map else 0),
        "spread":        float(mkt.get("bestAsk") or 0) - float(mkt.get("bestBid") or 0),
        "outcomes":      outcomes,
        "prices":        price_map,
        "token_ids":     token_ids,
        "event_slug":    mkt.get("slug", ""),
        "active":        mkt.get("active", False),
    }

print("Helper functions defined.")

Helper functions defined.


## 1. Pull Live Prices — Blue Wave Arb (Today's Case)
Three related markets:
| Market | Condition | Price |
|--------|-----------|-------|
| House Dem control | Dems ≥ 218 seats | **0.845** |
| Senate Dem control | Dems majority | **0.575** |
| Blue Wave | House Dem **AND** Senate Dem (≥49) | **0.875** 🚨 |

Since Blue Wave = House(D) ∩ Senate(D), we need: `P(Wave) ≤ P(House Dem) = 0.845`  
But 0.875 > 0.845 → **impossible by probability theory → risk-free arb exists**

In [13]:
SLUGS = {
    "house":  "which-party-will-win-the-house-in-2026",
    "senate": "which-party-will-win-the-senate-in-2026",
    "wave":   "blue-wave-in-2026",
}

# Pull live data and extract the relevant YES markets
live = {}
for key, slug in SLUGS.items():
    ev = fetch_event(slug)
    for mkt in ev.get("markets", []):
        vol = float(mkt.get("volume") or 0)
        if vol == 0:
            continue
        q = mkt.get("question", "")
        outcomes = json.loads(mkt.get("outcomes", "[]"))
        prices   = json.loads(mkt.get("outcomePrices", "[]"))
        price_map = dict(zip(outcomes, [float(p) for p in prices]))

        if key == "house" and "Yes" in price_map and "Democratic" in q:
            live["house_dem"] = {
                "label":     "House Dem control",
                "yes_last":  float(mkt.get("lastTradePrice") or price_map["Yes"]),
                "yes_bid":   float(mkt.get("bestBid") or 0),
                "yes_ask":   float(mkt.get("bestAsk") or 0),
                "volume":    vol,
                "liquidity": float(mkt.get("liquidity") or 0),
            }
        elif key == "senate" and "Yes" in price_map and "Democratic" in q:
            live["senate_dem"] = {
                "label":     "Senate Dem control",
                "yes_last":  float(mkt.get("lastTradePrice") or price_map["Yes"]),
                "yes_bid":   float(mkt.get("bestBid") or 0),
                "yes_ask":   float(mkt.get("bestAsk") or 0),
                "volume":    vol,
                "liquidity": float(mkt.get("liquidity") or 0),
            }
        elif key == "wave" and "Yes" in price_map:
            live["wave"] = {
                "label":     "Blue Wave (both)",
                "yes_last":  float(mkt.get("lastTradePrice") or price_map["Yes"]),
                "yes_bid":   float(mkt.get("bestBid") or 0),
                "yes_ask":   float(mkt.get("bestAsk") or 0),
                "volume":    vol,
                "liquidity": float(mkt.get("liquidity") or 0),
            }

rows = []
for k, v in live.items():
    rows.append({
        "Market":        v["label"],
        "Last":          v["yes_last"],
        "Bid (YES)":     v["yes_bid"],
        "Ask (YES)":     v["yes_ask"],
        "Spread":        round(v["yes_ask"] - v["yes_bid"], 4),
        "Volume $":      f"${v['volume']:,.0f}",
        "Liquidity $":   f"${v['liquidity']:,.0f}",
    })

df = pd.DataFrame(rows)
df

,Market,Last,Bid (YES),Ask (YES),Spread,Volume $,Liquidity $
0,House Dem control,0.85,0.84,0.85,0.01,"$2,276,328","$253,930"
1,Senate Dem control,0.58,0.56,0.58,0.02,"$1,037,682","$195,961"
2,Blue Wave (both),0.87,0.86,0.88,0.02,"$33,976","$22,663"


## 2. Arbitrage Analysis
### Logical constraint
Blue Wave resolves YES **only if** House(D) AND Senate(D) both resolve YES.  
Therefore by the monotonicity of probability:
$$P(\text{Wave}) \leq P(\text{House Dem}) \quad \text{and} \quad P(\text{Wave}) \leq P(\text{Senate Dem})$$

### Trade structure — guaranteed profit
| Leg | Action | Price | Cash flow |
|-----|--------|-------|-----------|
| Blue Wave NO | BUY | ask_NO = 1 − bid_YES | −(1−0.87) = **−0.13** |
| Senate Dem YES | BUY | ask_YES | **−0.58** |
| **Total cost** | | | **−0.71** |

| Scenario | Wave NO pays | Senate Dem YES pays | Net |
|----------|-------------|---------------------|-----|
| Senate=D, House=D (Wave=YES) | 0 | +1 | +1 → P&L **+0.29** |
| Senate=D, House=R (Wave=NO) | +1 | +1 | +2 → P&L **+1.29** |
| Senate=R, House=D (Wave=NO) | +1 | 0 | +1 → P&L **+0.29** |
| Senate=R, House=R (Wave=NO) | +1 | 0 | +1 → P&L **+0.29** |

**Minimum guaranteed return: +$0.29 on $0.71 invested = +40.8% in all scenarios**

In [4]:
hd   = live["house_dem"]
sd   = live["senate_dem"]
wave = live["wave"]

# Upper bound: Wave ≤ min(House Dem, Senate Dem) by probability monotonicity
bound = min(hd["yes_last"], sd["yes_last"])
gap   = wave["yes_last"] - bound

# Arb trade: buy Blue Wave NO (at ask_NO) + buy Senate Dem YES (at ask_YES)
# ask_NO = 1 - bid_YES  (since NO = complement of YES)
wave_no_ask  = 1 - wave["yes_bid"]    # cost to buy Blue Wave NO
senate_ask   = sd["yes_ask"]          # cost to buy Senate Dem YES
total_cost   = wave_no_ask + senate_ask

# Minimum payoff across all 4 scenarios
# Worst case: Wave=YES (Senate=D, House=D) → Wave NO pays 0, Senate YES pays 1 → net=1
min_payoff   = 1.0   # always get at least 1 (Senate YES or Wave NO pays)
min_profit   = min_payoff - total_cost
min_return   = min_profit / total_cost

# Max payoff: Wave=NO with Senate=D → both pay → net=2
max_profit   = 2.0 - total_cost

print(f"{'─'*58}")
print(f"  ARBITRAGE METRICS  (as of {AS_OF})")
print(f"{'─'*58}")
print(f"  House Dem YES last:      {hd['yes_last']:.4f}")
print(f"  Senate Dem YES last:     {sd['yes_last']:.4f}")
print(f"  Blue Wave YES last:      {wave['yes_last']:.4f}")
print(f"")
print(f"  Theoretical upper bound: {bound:.4f}  (min of House, Senate)")
print(f"  Mispricing gap:         {gap:+.4f}  ({gap*100:+.1f}pp)  {'🚨 ARB EXISTS' if gap > 0 else '✓ ok'}")
print(f"")
print(f"  TRADE: Buy Wave NO @ {wave_no_ask:.4f}  +  Buy Senate Dem YES @ {senate_ask:.4f}")
print(f"  Total cost per $1 notional: ${total_cost:.4f}")
print(f"")
print(f"  Worst-case profit:  ${min_profit:.4f}  ({min_return*100:.1f}% return)")
print(f"  Best-case profit:   ${max_profit:.4f}")
print(f"")
liq_cap = min(wave["liquidity"], sd["liquidity"])
print(f"  Capacity (min liquidity): ${liq_cap:,.0f}")
print(f"  Max $ profit at capacity: ${liq_cap * min_profit / total_cost:,.0f}")
print(f"{'─'*58}")

──────────────────────────────────────────────────────────
  ARBITRAGE METRICS  (as of 2026-04-16 09:39 UTC)
──────────────────────────────────────────────────────────
  House Dem YES last:      0.8400
  Senate Dem YES last:     0.5800
  Blue Wave YES last:      0.8700

  Theoretical upper bound: 0.5800  (min of House, Senate)
  Mispricing gap:         +0.2900  (+29.0pp)  🚨 ARB EXISTS

  TRADE: Buy Wave NO @ 0.1300  +  Buy Senate Dem YES @ 0.5800
  Total cost per $1 notional: $0.7100

  Worst-case profit:  $0.2900  (40.8% return)
  Best-case profit:   $1.2900

  Capacity (min liquidity): $26,104
  Max $ profit at capacity: $10,662
──────────────────────────────────────────────────────────


## 3. Scan: All Political Markets for Constrained Pairs
Look for any market where the **conjunction event** (e.g. "both A and B") trades above either individual component.  
Define known constraint pairs manually (from market knowledge) — this is the seed set for the signal.

In [5]:
# Known logical constraints: (conjunction_slug, [component_slugs], description)
# Format: conjunction market YES price must be <= ALL component YES prices
CONSTRAINED_PAIRS = [
    {
        "conjunction": ("blue-wave-in-2026",  "Blue Wave YES"),
        "components":  [
            ("which-party-will-win-the-house-in-2026",   "House Dem YES",   "Democratic"),
            ("which-party-will-win-the-senate-in-2026",  "Senate Dem YES",  "Democratic"),
        ],
        "note": "Wave = House(D) AND Senate(D≥49). Wave price must ≤ House Dem price.",
    },
    # Add more known pairs here as we find them, e.g.:
    # Red wave, trifecta, sweep markets etc.
]

def get_yes_price(slug, outcome_keyword=None):
    """Return (last, bid, ask, volume, liquidity) for YES (or first outcome matching keyword)."""
    ev = fetch_event(slug)
    for mkt in ev.get("markets", []):
        vol = float(mkt.get("volume") or 0)
        if vol == 0:
            continue
        outcomes   = json.loads(mkt.get("outcomes", "[]"))
        prices     = json.loads(mkt.get("outcomePrices", "[]"))
        price_map  = dict(zip(outcomes, [float(p) for p in prices]))
        q = mkt.get("question", "")
        if outcome_keyword and outcome_keyword not in q:
            continue
        if "Yes" in price_map:
            return {
                "last":      float(mkt.get("lastTradePrice") or price_map.get("Yes", 0)),
                "bid":       float(mkt.get("bestBid") or 0),
                "ask":       float(mkt.get("bestAsk") or 0),
                "volume":    vol,
                "liquidity": float(mkt.get("liquidity") or 0),
            }
    return None

arb_table = []
for pair in CONSTRAINED_PAIRS:
    conj_slug, conj_label = pair["conjunction"]
    conj_data = get_yes_price(conj_slug)
    if not conj_data:
        continue

    for comp_slug, comp_label, kw in pair["components"]:
        comp_data = get_yes_price(comp_slug, kw)
        if not comp_data:
            continue

        gap = conj_data["last"] - comp_data["last"]
        arb_table.append({
            "Conjunction":   conj_label,
            "Conj. Last":    conj_data["last"],
            "Conj. Bid":     conj_data["bid"],
            "Component":     comp_label,
            "Comp. Last":    comp_data["last"],
            "Comp. Ask":     comp_data["ask"],
            "Gap (C−M)":     round(gap, 4),
            "Arb?":          "🚨 YES" if gap > 0.005 else ("⚠ marginal" if gap > 0 else "✓ ok"),
            "Wave Liq $":    f"${conj_data['liquidity']:,.0f}",
            "Comp Liq $":    f"${comp_data['liquidity']:,.0f}",
            "Note":          pair["note"],
        })

arb_df = pd.DataFrame(arb_table)
arb_df

,Conjunction,Conj. Last,Conj. Bid,Component,Comp. Last,Comp. Ask,Gap (C−M),Arb?,Wave Liq $,Comp Liq $,Note
0,Blue Wave YES,0.87,0.87,House Dem YES,0.84,0.85,0.03,🚨 YES,"$26,104","$253,289",Wave = House(D) AND Senate(D≥49). Wave price m...
1,Blue Wave YES,0.87,0.87,Senate Dem YES,0.58,0.58,0.29,🚨 YES,"$26,104","$192,845",Wave = House(D) AND Senate(D≥49). Wave price m...


## 4. Signal Skeleton
This is the stub for `signals/polymk_arb.py` — daily scan that:
1. Runs the constraint check above  
2. Records any gap > 0 with timestamp  
3. Returns a signal value: `gap` (positive = arb exists, magnitude = attractiveness)

**Backtesting idea:** replay against historical price snapshots from the PRICES pipeline — measure how long mispricings persist, typical entry/exit spread, and whether they resolve profitably before contract expiry.

In [6]:
# Signal stub — daily snapshot record
# When formalised this moves to signals/polymk_arb.py

import pandas as pd
from datetime import datetime, timezone

def compute_arb_signal(pairs: list[dict]) -> pd.DataFrame:
    """
    For each constrained pair, compute the gap:
        gap = P(conjunction) - P(component)
    Positive gap = mispricing (conjunction > component, impossible).
    Returns a DataFrame row per pair with timestamp.
    """
    records = []
    ts = datetime.now(timezone.utc)
    for pair in pairs:
        conj_slug, conj_label = pair["conjunction"]
        conj = get_yes_price(conj_slug)
        if not conj:
            continue
        for comp_slug, comp_label, kw in pair["components"]:
            comp = get_yes_price(comp_slug, kw)
            if not comp:
                continue
            gap = conj["last"] - comp["last"]
            # Executable gap: accounting for bid/ask
            # Buy conj NO at (1 - conj_bid), buy comp YES at comp_ask
            exec_cost = (1 - conj["bid"]) + comp["ask"]
            exec_min_profit = 1.0 - exec_cost  # min payoff is always 1
            records.append({
                "timestamp":       ts,
                "conjunction":     conj_label,
                "component":       comp_label,
                "conj_last":       conj["last"],
                "comp_last":       comp["last"],
                "gap_last":        round(gap, 5),
                "exec_cost":       round(exec_cost, 4),
                "exec_min_profit": round(exec_min_profit, 4),
                "exec_min_return": round(exec_min_profit / exec_cost, 4) if exec_cost > 0 else None,
                "conj_liquidity":  conj["liquidity"],
                "comp_liquidity":  comp["liquidity"],
                "arb_flag":        gap > 0.005,
            })
    return pd.DataFrame(records)

signal_df = compute_arb_signal(CONSTRAINED_PAIRS)
signal_df

,timestamp,conjunction,component,conj_last,comp_last,gap_last,exec_cost,exec_min_profit,exec_min_return,conj_liquidity,comp_liquidity,arb_flag
0,2026-04-16 09:39:41.993428+00:00,Blue Wave YES,House Dem YES,0.87,0.84,0.03,0.98,0.02,0.0204,26104.3592,253289.1200,True
1,2026-04-16 09:39:41.993428+00:00,Blue Wave YES,Senate Dem YES,0.87,0.58,0.29,0.71,0.29,0.4085,26104.3592,192844.8575,True


## 5. Position Check — Current Open Trades
**Actual positions (as of 2026-04-16):**
| Leg | Shares | Avg Cost | Current | Value | P&L |
|-----|--------|----------|---------|-------|-----|
| Senate Dem YES | 298.2 | 57¢ | 57¢ | $169.93 | -$0.06 |
| Blue Wave NO | 1,203.9 | 14.1¢ | 12.5¢ | $150.48 | -$19.48 |

**⚠ Problem:** Equal *dollars* in each leg ≠ equal *shares*. Since each share pays exactly **$1** at resolution, the arb guarantee requires equal share counts, not equal dollar amounts.

In [14]:
# ── Current positions (from Polymarket UI) ──────────────────────────────────
pos_senate_yes_shares = 1204.1  # Senate Dem YES
pos_senate_yes_avg    = 0.578

pos_wave_no_shares    = 1203.9  # Blue Wave NO
pos_wave_no_avg       = 0.141

# Live prices now
senate_yes_bid = live["senate_dem"]["yes_bid"]   # sell here
senate_yes_ask = live["senate_dem"]["yes_ask"]   # buy here
wave_no_bid    = 1 - live["wave"]["yes_ask"]     # sell Wave NO = buy Wave YES at ask
wave_no_ask    = 1 - live["wave"]["yes_bid"]     # buy Wave NO = sell Wave YES at bid

# ── Why equal shares matter ──────────────────────────────────────────────────
# Each share pays exactly $1 if it resolves YES/NO.
# The arb payoff table (from cell 7) only works if Wave NO shares == Senate YES shares.
# With N shares of each:
#   Every scenario → you collect at least N × $1.00
# With mismatched sizes you have NAKED exposure on the excess.

target = max(pos_senate_yes_shares, pos_wave_no_shares)  # match up to the larger leg
shortfall_senate = target - pos_senate_yes_shares         # shares of Senate YES to BUY
excess_wave_no   = pos_wave_no_shares - pos_senate_yes_shares  # shares of Wave NO to SELL (alternative)

print(f"{'═'*62}")
print(f"  REBALANCE ANALYSIS")
print(f"{'═'*62}")
print(f"  Current Senate YES:  {pos_senate_yes_shares:>8.1f} shares")
print(f"  Current Wave   NO:   {pos_wave_no_shares:>8.1f} shares")
print(f"  Imbalance:           {excess_wave_no:>8.1f} extra Wave NO shares  ⚠")
print()

# Option A: Buy more Senate YES to match Wave NO (1,203.9 shares)
cost_buy_senate = shortfall_senate * senate_yes_ask
print(f"  OPTION A — Buy {shortfall_senate:.1f} Senate YES @ {senate_yes_ask:.3f}")
print(f"    Cost:            ${cost_buy_senate:>8.2f}")
print(f"    After:           {target:.1f} shares each → fully hedged")
print(f"    Guaranteed win:  ${target * (1.0 - (pos_wave_no_shares * wave_no_ask/target + pos_senate_yes_shares * senate_yes_ask/target)):,.0f}  (approx)")

# Option B: Sell excess Wave NO to match Senate YES (298.2 shares)
proceeds_sell_wave = excess_wave_no * wave_no_bid
new_target = pos_senate_yes_shares
print()
print(f"  OPTION B — Sell {excess_wave_no:.1f} Wave NO @ {wave_no_bid:.3f}")
print(f"    Proceeds:        ${proceeds_sell_wave:>8.2f}")
print(f"    After:           {new_target:.1f} shares each → fully hedged")
print(f"    Realised loss:   ${excess_wave_no * (wave_no_bid - pos_wave_no_avg):>8.2f}  (on the sold shares)")

# ── P&L on EXISTING position at current prices ───────────────────────────────
print()
print(f"{'─'*62}")
print(f"  CURRENT POSITION P&L (mark-to-market)")
print(f"{'─'*62}")
senate_pnl = pos_senate_yes_shares * (senate_yes_bid - pos_senate_yes_avg)
wave_pnl   = pos_wave_no_shares    * (wave_no_bid    - pos_wave_no_avg)
total_pnl  = senate_pnl + wave_pnl
print(f"  Senate YES: {pos_senate_yes_shares:.1f} sh × ({senate_yes_bid:.3f} − {pos_senate_yes_avg:.3f}) = ${senate_pnl:>+8.2f}")
print(f"  Wave   NO:  {pos_wave_no_shares:.1f} sh × ({wave_no_bid:.3f} − {pos_wave_no_avg:.3f}) = ${wave_pnl:>+8.2f}")
print(f"  Total MTM P&L:                                    ${total_pnl:>+8.2f}")

# ── Worst-case unhedged loss ─────────────────────────────────────────────────
print()
print(f"{'─'*62}")
print(f"  WORST-CASE SCENARIOS (unhedged excess Wave NO = {excess_wave_no:.0f} sh)")
print(f"{'─'*62}")
# Excess Wave NO shares are naked: if Wave resolves YES, they pay $0 (total loss)
naked_cost = excess_wave_no * pos_wave_no_avg
hedged_guarantee = pos_senate_yes_shares * 1.0 - (pos_senate_yes_shares * pos_senate_yes_avg + pos_senate_yes_shares * wave_no_ask)
print(f"  If Wave=YES (Dems win both):  naked {excess_wave_no:.0f} sh → lose ${naked_cost:,.2f}  ← MAX LOSS")
print(f"  If Wave=NO  (any other):      naked {excess_wave_no:.0f} sh → win  ${excess_wave_no:,.2f}")
print()
print(f"  → RECOMMENDATION: Option B (sell {excess_wave_no:.0f} Wave NO @ bid) is cheaper")
print(f"    and locks in the arb on the matched {new_target:.0f} share base.")

══════════════════════════════════════════════════════════════
  REBALANCE ANALYSIS
══════════════════════════════════════════════════════════════
  Current Senate YES:    1204.1 shares
  Current Wave   NO:     1203.9 shares
  Imbalance:               -0.2 extra Wave NO shares  ⚠

  OPTION A — Buy 0.0 Senate YES @ 0.580
    Cost:            $    0.00
    After:           1204.1 shares each → fully hedged
    Guaranteed win:  $337  (approx)

  OPTION B — Sell -0.2 Wave NO @ 0.120
    Proceeds:        $   -0.02
    After:           1204.1 shares each → fully hedged
    Realised loss:   $    0.00  (on the sold shares)

──────────────────────────────────────────────────────────────
  CURRENT POSITION P&L (mark-to-market)
──────────────────────────────────────────────────────────────
  Senate YES: 1204.1 sh × (0.560 − 0.578) = $  -21.67
  Wave   NO:  1203.9 sh × (0.120 − 0.141) = $  -25.28
  Total MTM P&L:                                    $  -46.96

───────────────────────────────────────